# Step 2.3 — Iceberg sandbox

Learn the mechanics on a throwaway table: snapshots, time travel, schema evolution, deletes.
The real raw-readings table is designed in Step 6.

Prereq: the Step 1 stack is running (`docker compose --env-file .env -f infra/docker-compose.yml up -d`).

In [1]:
from datetime import datetime, timedelta, timezone

import pyarrow as pa
from dotenv import find_dotenv, load_dotenv
from pyiceberg.expressions import LessThan
from pyiceberg.partitioning import PartitionField, PartitionSpec
from pyiceberg.schema import Schema
from pyiceberg.transforms import DayTransform
from pyiceberg.types import DoubleType, NestedField, StringType, TimestamptzType

load_dotenv(find_dotenv(usecwd=True))  # walks up from notebooks/ to the repo's .env

from murray.io.catalog import get_catalog

catalog = get_catalog()
TABLE = "sandbox.readings_demo"

## 1. Namespace and table

Re-runnable: drop and recreate the sandbox table every run.
Note the field IDs — Iceberg tracks columns by ID, not name. That is what makes renames
and schema evolution safe.

In [2]:
catalog.create_namespace_if_not_exists("sandbox")
if catalog.table_exists(TABLE):
    catalog.drop_table(TABLE)  # metadata only — files stay in S3 (see task 2.5)

schema = Schema(
    NestedField(1, "site_id", StringType(), required=True),
    NestedField(2, "ts", TimestamptzType(), required=True),  # timestamp WITH time zone, microseconds
    NestedField(3, "level_m", DoubleType(), required=False),
)
spec = PartitionSpec(
    PartitionField(source_id=2, field_id=1000, transform=DayTransform(), name="ts_day"),
)
tbl = catalog.create_table(TABLE, schema=schema, partition_spec=spec)
print(tbl.schema())
print(tbl.spec())

table {
  1: site_id: required string
  2: ts: required timestamptz
  3: level_m: optional double
}
[
  1000: ts_day: day(2)
]


In [3]:
def fake_readings(day: datetime, start_level: float, hours: int = 24, **extra_cols) -> pa.Table:
    """Hourly fake Barmah readings for one day, typed to match the table's *current* schema."""
    rows = [
        {"site_id": "409215", "ts": day + timedelta(hours=h), "level_m": round(start_level + 0.01 * h, 3),
         **{k: v for k, v in extra_cols.items()}}
        for h in range(hours)
    ]
    # Build with the table's Arrow schema so required/nullable and timestamp units line up.
    return pa.Table.from_pylist(rows, schema=tbl.schema().as_arrow())

day1 = datetime(2026, 10, 1, tzinfo=timezone.utc)
day2 = day1 + timedelta(days=1)

## 2. Two appends → two snapshots

In [4]:
tbl.append(fake_readings(day1, start_level=2.50))
tbl.append(fake_readings(day2, start_level=2.80))

tbl.inspect.snapshots().to_pandas()[["committed_at", "snapshot_id", "parent_id", "operation", "summary"]]

,committed_at,snapshot_id,parent_id,operation,summary
0,2026-10-23 06:03:52.726,8320666073289863869,NaN,append,"[(added-files-size, 1656), (added-data-files, ..."
1,2026-10-23 06:03:52.771,8528540704852771646,8.320666e+18,append,"[(added-files-size, 1666), (added-data-files, ..."


## 3. Time travel

Each snapshot is a complete, immutable version of the table. Reading an old one just
follows an old manifest list — no data is copied.

In [5]:
first_snapshot = tbl.snapshots()[0].snapshot_id
old = tbl.scan(snapshot_id=first_snapshot).to_pandas()
now = tbl.scan().to_pandas()
print(f"first snapshot: {len(old)} rows | current: {len(now)} rows")

first snapshot: 24 rows | current: 48 rows


## 4. Schema evolution

Add a nullable `flow_mld` column. This is a metadata-only change: no Parquet file is rewritten.

In [6]:
with tbl.update_schema() as update:
    update.add_column("flow_mld", DoubleType(), doc="Flow in megalitres/day")

day3 = day2 + timedelta(days=1)
tbl.append(fake_readings(day3, start_level=3.10, flow_mld=8200.0))

print(tbl.schema())

table {
  1: site_id: required string
  2: ts: required timestamptz
  3: level_m: optional double
  4: flow_mld: optional double (Flow in megalitres/day)
}


In [7]:
# Current read: old rows have no value for flow_mld, so they come back as null.
current = tbl.scan().to_pandas()
current.groupby(current.ts.dt.date)["flow_mld"].agg(["count", "mean"])

,count,mean
ts,,
2026-10-01,0,NaN
2026-10-02,0,NaN
2026-10-03,24,8200.0


In [8]:
# Old snapshot: PyIceberg reads it with the schema that was current *at that snapshot*,
# so flow_mld is not even a column here.
tbl.scan(snapshot_id=first_snapshot).to_pandas().columns.tolist()

['site_id', 'ts', 'level_m']

## 5. Delete

Deleting rows creates a new snapshot. Whole files that match are dropped; partly-matching
files are rewritten (copy-on-write). Old files stay reachable from old snapshots.

In [9]:
files_before = set(tbl.inspect.files().to_pandas().file_path)
tbl.delete(delete_filter=LessThan("level_m", 2.60))  # removes the first 10 hours of day 1
files_after = set(tbl.inspect.files().to_pandas().file_path)

print("rows now:", len(tbl.scan().to_pandas()))
print("files removed from current snapshot:", len(files_before - files_after))
print("files added:", len(files_after - files_before))
tbl.inspect.snapshots().to_pandas()[["committed_at", "snapshot_id", "operation"]]

rows now: 62
files removed from current snapshot: 1
files added: 1


,committed_at,snapshot_id,operation
0,2026-10-23 06:03:52.726,8320666073289863869,append
1,2026-10-23 06:03:52.771,8528540704852771646,append
2,2026-10-23 06:03:52.875,5947536011690803706,append
3,2026-10-23 06:03:52.978,4868061544572075778,overwrite


## 6. Under the hood (task 2.4)

### 6a. The catalog row

The SQL catalog is just a table in Postgres. For each Iceberg table it stores **one pointer**:
the current `metadata_location`. A commit = writing a new metadata.json to S3, then swapping
this pointer in one SQL transaction (only if it still equals what the writer started from).

In [10]:
import os

import pandas as pd
import psycopg


def catalog_rows() -> pd.DataFrame:
    with psycopg.connect(
        host=os.environ.get("POSTGRES_HOST", "localhost"),
        port=os.environ.get("POSTGRES_PORT", "5432"),
        user="iceberg_catalog",
        password=os.environ["ICEBERG_DB_PASSWORD"],
        dbname="iceberg_catalog",
    ) as conn:
        cur = conn.execute(
            "SELECT table_namespace, table_name, metadata_location, previous_metadata_location"
            " FROM iceberg_tables"
        )
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])


before = catalog_rows()
before

,table_namespace,table_name,metadata_location,previous_metadata_location
0,sandbox,readings_demo,s3://lakehouse/warehouse/sandbox/readings_demo...,s3://lakehouse/warehouse/sandbox/readings_demo...


In [11]:
# One more commit, then compare: the pointer moved and the old one became "previous".
tbl.append(fake_readings(day3 + timedelta(days=1), start_level=3.20, flow_mld=8300.0))
after = catalog_rows()

print("before :", before.metadata_location[0].rsplit("/", 1)[-1])
print("after  :", after.metadata_location[0].rsplit("/", 1)[-1])
print("previous now points at the old one:",
      after.previous_metadata_location[0] == before.metadata_location[0])

before : 00005-ec4f193a-f698-4786-ad70-c5760bfaa817.metadata.json
after  : 00006-9085c5ac-9b17-40b1-8663-c46e65bd264a.metadata.json
previous now points at the old one: True


### 6b. The files in S3, layer by layer

In [12]:
from murray.io.s3 import delete_prefix, list_keys

TABLE_PREFIX = tbl.location().removeprefix("s3://lakehouse/") + "/"


def layer(key: str) -> str:
    name = key.rsplit("/", 1)[-1]
    if name.endswith(".metadata.json"):
        return "1 table metadata (metadata.json)"
    if name.startswith("snap-") and name.endswith(".avro"):
        return "2 manifest list (snap-*.avro)"
    if name.endswith(".avro"):
        return "3 manifest (*-m*.avro)"
    if name.endswith(".parquet"):
        return "4 data file (.parquet)"
    return "other"


keys = list_keys(TABLE_PREFIX)
print("table location:", tbl.location())
pd.Series([layer(k) for k in keys]).value_counts().sort_index()

table location: s3://lakehouse/warehouse/sandbox/readings_demo


1 table metadata (metadata.json)    7
2 manifest list (snap-*.avro)       5
3 manifest (*-m*.avro)              6
4 data file (.parquet)              5
Name: count, dtype: int64

Why so many files for one small table? Every commit writes a new metadata.json and a new
manifest list, and old ones are never overwritten (files are immutable). If you have re-run
this notebook, there are also leftovers from previous runs — see section 7.

Now follow the chain for the **current** snapshot only:

In [13]:
snap = tbl.current_snapshot()
manifests = snap.manifests(tbl.io)
data_files = tbl.inspect.files().to_pandas().file_path.tolist()

print("catalog  →", after.metadata_location[0].removeprefix(tbl.location()))
print("snapshot →", snap.snapshot_id, f"({snap.summary.operation.value})")
print("manifest list →", snap.manifest_list.removeprefix(tbl.location()))
for m in manifests:
    print("   manifest →", m.manifest_path.removeprefix(tbl.location()),
          f"(added={m.added_files_count}, existing={m.existing_files_count}, deleted={m.deleted_files_count})")
for f in data_files:
    print("      data →", f.removeprefix(tbl.location()))
print(f"\n{len(data_files)} live data files out of "
      f"{sum(k.endswith('.parquet') for k in keys)} Parquet files under the table prefix")

catalog  → /metadata/00006-9085c5ac-9b17-40b1-8663-c46e65bd264a.metadata.json
snapshot → 7560475134454576137 (append)
manifest list → /metadata/snap-7560475134454576137-0-15f163ab-23bc-4c46-b3ba-aac964eca460.avro
   manifest → /metadata/15f163ab-23bc-4c46-b3ba-aac964eca460-m0.avro (added=1, existing=0, deleted=0)
   manifest → /metadata/31c3a5f5-a80b-4b3b-9632-c0da3c18c052-m0.avro (added=1, existing=0, deleted=0)
   manifest → /metadata/a2fa5845-820d-4748-838d-cff697b653b3-m0.avro (added=1, existing=0, deleted=0)
   manifest → /metadata/79353c87-5ebf-422e-a649-6975c0186386-m0.avro (added=1, existing=0, deleted=0)
      data → /data/ts_day=2026-10-04/00000-0-15f163ab-23bc-4c46-b3ba-aac964eca460.parquet
      data → /data/ts_day=2026-10-01/00000-0-31c3a5f5-a80b-4b3b-9632-c0da3c18c052.parquet
      data → /data/ts_day=2026-10-03/00000-0-a2fa5845-820d-4748-838d-cff697b653b3.parquet
      data → /data/ts_day=2026-10-02/00000-0-79353c87-5ebf-422e-a649-6975c0186386.parquet

4 live data files 

Note the `data/ts_day=2026-10-0X/` folders: that is the `day(ts)` partition. A query for
"last 7 days" skips whole partitions from the manifest's partition values, then skips files
inside a partition using each file's min/max column stats — all before opening any Parquet.

In [14]:
# The manifests also store per-file column stats used for pruning:
tbl.inspect.files().to_pandas()[["partition", "record_count", "lower_bounds", "upper_bounds"]]

,partition,record_count,lower_bounds,upper_bounds
0,{'ts_day': 2026-10-04},24,"[(1, b'409215'), (2, b'\x00\xc0Sr\xf8\\\x06\x0...","[(1, b'409215'), (2, b'\x00|\x97\xb9\x0b]\x06\..."
1,{'ts_day': 2026-10-01},14,"[(1, b'409215'), (2, b'\x00\x08\x92z\xc4\\\x06...","[(1, b'409215'), (2, b'\x00\\\x11`\xcf\\\x06\x..."
2,{'ts_day': 2026-10-03},24,"[(1, b'409215'), (2, b'\x00`|T\xe4\\\x06\x00')...","[(1, b'409215'), (2, b'\x00\x1c\xc0\x9b\xf7\\\..."
3,{'ts_day': 2026-10-02},24,"[(1, b'409215'), (2, b'\x00\x00\xa56\xd0\\\x06...","[(1, b'409215'), (2, b'\x00\xbc\xe8}\xe3\\\x06..."


## 7. Drop vs purge (task 2.5)

In [15]:
n_before = len(list_keys(TABLE_PREFIX))
catalog.drop_table(TABLE)
n_after = len(list_keys(TABLE_PREFIX))

print("catalog rows for the table:", len(catalog_rows()))
print(f"objects under the table prefix: {n_before} before drop, {n_after} after drop")

catalog rows for the table: 0
objects under the table prefix: 23 before drop, 23 after drop


`drop_table` only removes the catalog row. The files stay because the catalog cannot know
whether anything else still points at them (another catalog registered on the same files,
a branch/tag, a backup, an in-flight reader). Deleting data is a separate, deliberate step.

Options:
- `catalog.purge_table(name)` — drop **and** delete every file reachable from the table's
  snapshots. It cannot see files left by *earlier* tables at the same location.
- `tbl.maintenance.expire_snapshots()` — on a live table, forget old snapshots and delete files
  only they referenced. This is what bounds storage growth (and limits time travel).
- Orphan-file cleanup — delete files under the location that no metadata references at all.

The demo below recreates a small table at the **same location** and purges it:

In [16]:
tbl = catalog.create_table(TABLE, schema=schema, partition_spec=spec)
tbl.append(fake_readings(day1, start_level=2.50))
n_with_new = len(list_keys(TABLE_PREFIX))
catalog.purge_table(TABLE)
n_after_purge = len(list_keys(TABLE_PREFIX))
print(f"objects: {n_with_new} with new table, {n_after_purge} after purge_table")
print("→ purge deleted every file the new table's metadata knew about. The files left over "
      "belong to the earlier, dropped table: orphans that nothing references, so nothing deletes them.")

objects: 28 with new table, 23 after purge_table
→ purge deleted every file the new table's metadata knew about. The files left over belong to the earlier, dropped table: orphans that nothing references, so nothing deletes them.


In [17]:
# Clean up the orphans so the next run starts fresh. Safe here: the table is gone and
# the prefix belongs to the sandbox only. Never do this under a live table's location.
print("deleted", delete_prefix(TABLE_PREFIX), "orphan files")

deleted 23 orphan files
